In [1]:
import sys
from pathlib import Path

In [2]:
REPO_ROOT = Path.cwd().resolve().parents[2]

In [3]:
REPO_ROOT

WindowsPath("C:/Users/YazeedMashaikh/Desktop/Jabarti_Yazeed's_Version")

In [4]:
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [5]:
from jabart_llm.tokenizer import Tokenizer
from jabarti.tokenizer_building.phases.build_tokenizer import ScratchTokenizer

In [6]:
TOKENIZER_PATH = Path(__name__) / "tokenizer.json"

tokenizer = Tokenizer(ScratchTokenizer)

tok = tokenizer.load_tokenizer()
print(f"Vocabulary Size: {tok.vocab_size}")

Vocabulary Size: 32000


In [7]:
sentence = "محمد    أحمد حماد المؤيدي؟؟"
print(f"Test Normalized: {tok.normalize(sentence)}")

Test Normalized: محمد احمد حماد المؤيدي؟؟


In [8]:
print(f"Test Normalized: {tok.encode(sentence)}")

Test Normalized: [8157, 8418, 9567, 7754, 9273, 28186, 1168, 1168]


In [9]:
ids = [8157, 8418, 9567, 7754, 9273, 28186, 1168, 1168]
print(f"Test Normalized: {tok.decode(ids)}")

Test Normalized: محمد احمد حماد المؤيدي؟؟


In [10]:
print(f"Test Normalized: {tok.tokenize(sentence)}")

Test Normalized: ['▁محمد', '▁احمد', '▁حم', 'اد', '▁المؤ', 'يدي', '؟', '؟']


## Normal Positional Embeddings

In [11]:
from jabart_llm.config import CONFIG

print(CONFIG)

ModelConfig(vocab_size=32000, d_model=768, max_seq_len=1024, dropout=0.1)


In [12]:
from jabart_llm.model.embeddings import InputEmbeddings

input_embeddings = InputEmbeddings(config=CONFIG)

In [13]:
tok = tokenizer.load_tokenizer()

In [14]:
Sentence_1= "Yazeed Mshayekh, يزيد    مشايخ"
Sentence_2= "Yazeed Mshayekh, يزيد    مشايخ"

In [15]:
Sentence_1_ids = tok.encode(Sentence_1)
Sentence_2_ids = tok.encode(Sentence_2)

print(f"Sentence 1 ids: {Sentence_1_ids} \nSentence 2 ids: {Sentence_2_ids}")

Sentence 1 ids: [13630, 14668, 7702, 9030, 7785, 9184, 13538, 14627, 8892, 1173, 8960] 
Sentence 2 ids: [13630, 14668, 7702, 9030, 7785, 9184, 13538, 14627, 8892, 1173, 8960]


In [16]:
import torch

input_ids = torch.tensor([Sentence_1_ids, Sentence_2_ids])

encode_input = input_embeddings.forward(input_ids=input_ids, position_offset=2, debug=True)

  input_ids       : [[13630, 14668, 7702, 9030, 7785, 9184, 13538, 14627, 8892, 1173, 8960], [13630, 14668, 7702, 9030, 7785, 9184, 13538, 14627, 8892, 1173, 8960]]  shape=(2, 11)
  position_offset : 2
  positions       : [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
  tok shape       : (2, 11, 768)   (B, T, d_model)
  pos shape       : (11, 768)      (T, d_model)
  pos == table rows [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] ? True
  first pos vector (first 4 values): [-0.10556033998727798, 0.16024257242679596, 0.5438308119773865, -0.27124181389808655]


## Sinusodial Positional Embeddings

In [17]:
from jabart_llm.model.sinusoidal_embeddings import InputEmbeddings

sin_input_embeddings = InputEmbeddings(config=CONFIG)

In [20]:
encode_input = sin_input_embeddings.forward(input_ids=input_ids, position_offset=2, debug=True)

FORWARD PASS
[1] input_ids       : [[13630, 14668, 7702, 9030, 7785, 9184, 13538, 14627, 8892, 1173, 8960], [13630, 14668, 7702, 9030, 7785, 9184, 13538, 14627, 8892, 1173, 8960]]  shape=(2, 11)  (B, T)
[2] position_offset : 2
    rows read from pe: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
[3] tok = token_embedding(input_ids)   shape=(2, 11, 768)  (B, T, d_model)
    LEARNED: random at start, changes during training. Sentence 0:
tensor([[ 0.013, -0.117, -0.680,  ...,  2.304, -2.616, -1.037],
        [-0.849,  0.522,  0.934,  ...,  1.187, -0.234,  0.212],
        [ 1.492,  0.111,  0.273,  ...,  0.057,  1.116,  1.482],
        ...,
        [-1.145, -0.396, -0.408,  ...,  0.540,  1.605,  1.395],
        [-0.094,  0.937, -1.197,  ...,  0.531,  1.170, -0.112],
        [-1.276, -1.112,  1.079,  ...,  0.052,  1.072,  0.926]], grad_fn=<SelectBackward0>)
[4] pos = pe[2:13]   shape=(11, 768)  (T, d_model)
    FIXED: identical every run, never trained:
tensor([[ 0.909, -0.416,  0.928,  ...,  1.000,  